**You are the average of the five people you spend the most time with.    - Jim Rohn**

The Quote: Suggests that an individual is shaped by the influences of the people closest to them.
k-NN Algorithm: Determines the class or value of a data point based on the characteristics of its closest neighbors in the feature space.
In both cases, proximity to a group influences the outcome:

In the quote, proximity is in terms of relationships or time spent.
In k-NN, proximity is in terms of Euclidean distance (or another metric) in a feature space.

Libraries

In [90]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import OneHotEncoder

In [91]:
data = pd.read_excel("C:/Users/wwwde/OneDrive - Florida Atlantic University/Academics/Research/NDHS/Dataset/Children's Recode/Children Recode.xlsx")
data.head()

,caseid,bidx,v000,v001,v002,v003,v004,v005,v006,v007,...,s604c,s607ea,s607eb,s607ec,s607ed,s607ex,s607f,s607g,s615f,s631o
0,1 1 2,1,NP8,1,1,2,1,916093,10,2078,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
1,1 8 2,1,NP8,1,8,2,1,916093,10,2078,...,0.0,0.0,1.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0
2,1 8 2,2,NP8,1,8,2,1,916093,10,2078,...,NaN,0.0,1.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0
3,1 9 3,1,NP8,1,9,3,1,916093,10,2078,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,NaN,0.0
4,1 19 1,1,NP8,1,19,1,1,916093,10,2078,...,0.0,1.0,1.0,0.0,0.0,0.0,NaN,0.0,NaN,NaN


In [92]:
# Select columns of interest
df1 = data.copy()
df1 = df1[['b19', 'v106', 'v190', 'v102', 'v101', 'v445', 'b11', 'v137', 'v130', 'v131', 'm19', 'b4', 'm4', 'v012', 'v729', 'v714', 'v481', 'hw71', 'hw70']]
df1.rename(columns={'b19': 'Child_age', 'v106':'Mother_education', 'v190':'Wealth_index', 'v102':'Place_residence', 'v101':'Region', 'v445':'BMI', 'b11':'Birth_interval', 'v137':'Children_under5', 'v130':'Religion', 'v131':'Ethnicity', 'm19':'Birth_weight', 'b4':'Child_sex', 'm4':'Breastfeeding_duration', 'v012':'Mother_age', 'v729':'Father_education', 'v714':'Mother_working', 'v481':'Health_insurance', 'hw71':'Status'}, inplace=True)

# Drop rows with encoded values
df2 = df1.copy()
df2.drop(index = 3251, inplace=True) # BMI 9998.0
df2.drop(index = [658,730, 4417], inplace = True) # Religion 96
df2.drop(index = [1369, 602, 603], inplace=True) # Ethnicity 96
df2 = df2[~df2['Birth_weight'].isin([9998, 9996])] # Birth_weight 9998 or 9996
df2.drop(index = 1651, inplace=True) # Status 9998

#Drop column to get maximum possible observations
df3 = df2.drop(columns= ['hw70', 'Birth_interval', 'Birth_weight', 'Breastfeeding_duration'])

# Drop rows with missing values
df4=df3.dropna().copy()

#Actual range of values
df4['BMI'] = df4['BMI']/100
df4['Status'] = df4['Status']/100
df4['Status'] = [0 if -2 <= val <= 2 else 1 for val in df4['Status']]
df4.head()

,Child_age,Mother_education,Wealth_index,Place_residence,Region,BMI,Children_under5,Religion,Ethnicity,Child_sex,Mother_age,Father_education,Mother_working,Health_insurance,Status
3,17,1,1,2,1,22.00,1,1,2,2,34,3.0,0,0,0
4,40,2,2,2,1,25.10,2,4,8,1,26,2.0,1,0,0
5,59,2,2,2,1,25.10,2,4,8,2,26,2.0,1,0,0
11,55,2,2,2,1,21.53,1,4,8,2,28,1.0,1,0,1
14,14,1,1,2,1,28.03,1,4,8,1,26,0.0,1,0,0


In [93]:
# Save the cleaned dataset
df4.to_csv("C:/Users/wwwde/OneDrive - Florida Atlantic University/Academics/Research/NDHS/Dataset/Children's Recode/Children Recode_cleaned.csv", index=False)

In [94]:
#Train-Test Split
X = df4.iloc[:,:-1]
y = df4.iloc[:,-1]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

## Model fitting

In [115]:
knn = KNeighborsClassifier(n_neighbors=10)
knn.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=10)

In [116]:
predictions = knn.predict(X_test)
accuracy = knn.score(X_test, y_test)
print(accuracy)

0.8159645232815964


### Standarize the data

In [97]:
columns_to_scale = ['Child_age', 'BMI', 'Mother_age']
scaler = StandardScaler()
X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])
X_test[columns_to_scale] = scaler.transform(X_test[columns_to_scale])


In [98]:
X_train.head()

,Child_age,Mother_education,Wealth_index,Place_residence,Region,BMI,Children_under5,Religion,Ethnicity,Child_sex,Mother_age,Father_education,Mother_working,Health_insurance
287,-0.051559,1,4,1,1,-0.816583,1,1,2,2,0.497648,2.0,0,0
377,0.931298,1,2,1,1,-0.373358,1,1,8,1,-0.644481,3.0,1,1
2972,0.295332,0,1,2,5,-0.066509,1,1,2,1,2.020487,3.0,1,1
3165,0.410962,1,1,2,5,-0.276320,2,1,8,2,-0.644481,1.0,1,0
2427,-1.439121,2,2,1,3,1.863752,1,2,8,1,-1.025191,3.0,0,0


In [119]:
knn_scaled = KNeighborsClassifier(n_neighbors=10)
knn_scaled.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=10)

In [120]:
predictions_scaled = knn_scaled.predict(X_test)
accuracy_scaled = knn_scaled.score(X_test, y_test)
print(accuracy_scaled)

0.8159645232815964


### Using one-hot encoding to nominal columns

In [104]:
# Select columns to be encoded
columns_to_encode = ['Place_residence', 'Region', 'Religion', 'Ethnicity', 'Child_sex', 'Mother_working', 'Health_insurance']

# Initialize the OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit and transform the training data
X_train_encoded = encoder.fit_transform(X_train[columns_to_encode])
X_test_encoded = encoder.transform(X_test[columns_to_encode])

# Create DataFrames from the encoded data
X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=encoder.get_feature_names_out(columns_to_encode))
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=encoder.get_feature_names_out(columns_to_encode))

# Drop the original categorical columns and concatenate the encoded columns
X_train = X_train.drop(columns=columns_to_encode).reset_index(drop=True)
X_test = X_test.drop(columns=columns_to_encode).reset_index(drop=True)
X_train = pd.concat([X_train, X_train_encoded_df], axis=1)
X_test = pd.concat([X_test, X_test_encoded_df], axis=1)


In [108]:
X_train.head()

,Child_age,Mother_education,Wealth_index,BMI,Children_under5,Mother_age,Father_education,Place_residence_1,Place_residence_2,Region_1,...,Ethnicity_7,Ethnicity_8,Ethnicity_9,Ethnicity_10,Child_sex_1,Child_sex_2,Mother_working_0,Mother_working_1,Health_insurance_0,Health_insurance_1
0,-0.051559,1,4,-0.816583,1,0.497648,2.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0
1,0.931298,1,2,-0.373358,1,-0.644481,3.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
2,0.295332,0,1,-0.066509,1,2.020487,3.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
3,0.410962,1,1,-0.276320,2,-0.644481,1.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0
4,-1.439121,2,2,1.863752,1,-1.025191,3.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0


In [121]:
knn_encoded = KNeighborsClassifier(n_neighbors=10)
knn_encoded.fit(X_train, y_train)


KNeighborsClassifier(n_neighbors=10)

In [122]:
predictions_encoded = knn_encoded.predict(X_test)
accuracy_encoded = knn_encoded.score(X_test, y_test)
print(accuracy_encoded)

0.8159645232815964


**Work remained**
- Classification plot
- Find the appropriate k value